# 🏗️ Step 1.5: DINOv2 建物分類ヘッド学習

```
建物タイプ別の写真 (各10〜20枚)
  ↓ DINOv2 (frozen) で特徴抽出
384次元ベクトル
  ↓ 分類ヘッド学習 (数秒)
classifier.onnx
  ↓
step2_persona_train.ipynb で読み込み
  → 「牛丼屋らしさ」スコアをポリシーに渡す
  → HTML でリアルタイム表示
```

## 建物クラス
| ID | クラス | 例 |
|----|--------|----|
| 0 | gyudon | 牛丼屋・定食屋 |
| 1 | ramen | ラーメン屋 |
| 2 | bento | 弁当屋・惣菜 |
| 3 | cafe | カフェ・喫茶店 |
| 4 | office | オフィスビル |
| 5 | house | 住宅・マンション |
| 6 | conbini | コンビニ |
| 7 | hospital | 病院・クリニック |

**Runtime: T4 GPU 推奨 (CPUでも動作可)**

In [ ]:
# ============================================================
# セル 1: インストール
# ============================================================
!pip install torch torchvision onnx onnxscript onnxruntime \
             requests Pillow tqdm -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & 設定
# ============================================================
import torch, torch.nn as nn, onnx
import numpy as np
import os, json, io, time
from pathlib import Path
from PIL import Image
import requests
from tqdm import tqdm
import torchvision.transforms as T
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({
    'figure.facecolor':'#0a0c10','axes.facecolor':'#12161e',
    'text.color':'#c8d8e8','axes.labelcolor':'#c8d8e8',
})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mesa_persona'
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

# 建物クラス定義 (HTMLの BLDG_TYPES と同じ順番)
CLASSES = [
    'gyudon',    # 0: 牛丼屋・定食屋
    'ramen',     # 1: ラーメン屋
    'bento',     # 2: 弁当屋
    'cafe',      # 3: カフェ
    'office',    # 4: オフィスビル
    'house',     # 5: 住宅
    'conbini',   # 6: コンビニ
    'hospital',  # 7: 病院
]
N_CLASSES = len(CLASSES)
DINO_DIM  = 384  # dinov2_vits14 の CLS token 次元

print(f'クラス数: {N_CLASSES}')
print(f'DINOv2 出力次元: {DINO_DIM}')

In [ ]:
# ============================================================
# セル 3: 学習画像の準備
#
# 【方法A】 Unsplashの写真を自動ダウンロード (簡単)
# 【方法B】 自分で用意した画像をGoogle Driveにアップロード
#
# 方法Bの場合:
#   /content/drive/MyDrive/mesa_building_images/
#     gyudon/   img_001.jpg, img_002.jpg ...
#     ramen/    img_001.jpg ...
#     ...
# ============================================================

# Unsplash から各クラスの写真をダウンロード
# ※ Unsplash の写真は商用利用可・帰属表示推奨
# ※ 本番用途では実際の店舗写真を使うことを推奨

UNSPLASH_URLS = {
    'gyudon': [
        'https://images.unsplash.com/photo-1569718212165-3a8278d5f624?w=224&q=80',
        'https://images.unsplash.com/photo-1603133872878-684f208fb84b?w=224&q=80',
        'https://images.unsplash.com/photo-1547592166-23ac45744acd?w=224&q=80',
        'https://images.unsplash.com/photo-1528360983277-13d401cdc186?w=224&q=80',
        'https://images.unsplash.com/photo-1467003909585-2f8a72700288?w=224&q=80',
    ],
    'ramen': [
        'https://images.unsplash.com/photo-1569050467447-ce54b3bbc37d?w=224&q=80',
        'https://images.unsplash.com/photo-1557872943-16a5ac26437e?w=224&q=80',
        'https://images.unsplash.com/photo-1591814468924-caf88d1232e1?w=224&q=80',
        'https://images.unsplash.com/photo-1617196034183-421b4040ed20?w=224&q=80',
        'https://images.unsplash.com/photo-1511690743698-d9d85f2fbf38?w=224&q=80',
    ],
    'bento': [
        'https://images.unsplash.com/photo-1596797038530-2c107229654b?w=224&q=80',
        'https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=224&q=80',
        'https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=224&q=80',
        'https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=224&q=80',
        'https://images.unsplash.com/photo-1555939594-58d7cb561ad1?w=224&q=80',
    ],
    'cafe': [
        'https://images.unsplash.com/photo-1554118811-1e0d58224f24?w=224&q=80',
        'https://images.unsplash.com/photo-1501339847302-ac426a4a7cbb?w=224&q=80',
        'https://images.unsplash.com/photo-1559925393-8be0ec4767c8?w=224&q=80',
        'https://images.unsplash.com/photo-1445116572660-236099ec97a0?w=224&q=80',
        'https://images.unsplash.com/photo-1521017432531-fbd92d768814?w=224&q=80',
    ],
    'office': [
        'https://images.unsplash.com/photo-1486325212027-8081e485255e?w=224&q=80',
        'https://images.unsplash.com/photo-1497366216548-37526070297c?w=224&q=80',
        'https://images.unsplash.com/photo-1497366754035-f200968a6e72?w=224&q=80',
        'https://images.unsplash.com/photo-1541746972996-4e0b0f43e02a?w=224&q=80',
        'https://images.unsplash.com/photo-1504384308090-c894fdcc538d?w=224&q=80',
    ],
    'house': [
        'https://images.unsplash.com/photo-1568605114967-8130f3a36994?w=224&q=80',
        'https://images.unsplash.com/photo-1570129477492-45c003edd2be?w=224&q=80',
        'https://images.unsplash.com/photo-1558618666-fcd25c85cd64?w=224&q=80',
        'https://images.unsplash.com/photo-1583608205776-bfd35f0d9f83?w=224&q=80',
        'https://images.unsplash.com/photo-1523217582562-09d0def993a6?w=224&q=80',
    ],
    'conbini': [
        'https://images.unsplash.com/photo-1556742049-0cfed4f6a45d?w=224&q=80',
        'https://images.unsplash.com/photo-1563013544-824ae1b704d3?w=224&q=80',
        'https://images.unsplash.com/photo-1604719312566-8912e9227c6a?w=224&q=80',
        'https://images.unsplash.com/photo-1601598851547-4302969d0614?w=224&q=80',
        'https://images.unsplash.com/photo-1472851294608-062f824d29cc?w=224&q=80',
    ],
    'hospital': [
        'https://images.unsplash.com/photo-1519494026892-80bbd2d6fd0d?w=224&q=80',
        'https://images.unsplash.com/photo-1586773860418-d37222d8fce3?w=224&q=80',
        'https://images.unsplash.com/photo-1538108149393-fbbd81895907?w=224&q=80',
        'https://images.unsplash.com/photo-1581595219315-a187dd40c322?w=224&q=80',
        'https://images.unsplash.com/photo-1587351021759-3e566b3db4f1?w=224&q=80',
    ],
}

# 画像保存先
IMG_ROOT = Path(SAVE_DIR) / 'building_images'
IMG_ROOT.mkdir(exist_ok=True)

def download_images():
    """Unsplashから画像をダウンロードして保存"""
    for cls_name, urls in UNSPLASH_URLS.items():
        cls_dir = IMG_ROOT / cls_name
        cls_dir.mkdir(exist_ok=True)
        for i, url in enumerate(tqdm(urls, desc=cls_name, leave=False)):
            fpath = cls_dir / f'img_{i:03d}.jpg'
            if fpath.exists(): continue
            try:
                r = requests.get(url, timeout=10)
                r.raise_for_status()
                Image.open(io.BytesIO(r.content)).convert('RGB').save(fpath)
            except Exception as e:
                print(f'  ⚠ {cls_name}/{i}: {e}')

print('画像をダウンロード中...')
download_images()

# 確認
print('\n各クラスの画像数:')
total = 0
for cls in CLASSES:
    n = len(list((IMG_ROOT / cls).glob('*.jpg')))
    total += n
    print(f'  {cls:12s}: {n}枚')
print(f'  合計: {total}枚')

In [ ]:
# ============================================================
# セル 4: DINOv2 ロード & 特徴抽出
# ============================================================

# DINOv2 をロード (frozen)
print('DINOv2 (dinov2_vits14) をロード中...')
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True)
dino = dino.to(DEVICE).eval()
for p in dino.parameters(): p.requires_grad = False
print(f'✓ DINOv2 loaded ({sum(p.numel() for p in dino.parameters()):,} params, frozen)')

# ImageNet 正規化
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def extract_features(img_paths):
    """画像リストから DINOv2 特徴ベクトルを一括抽出"""
    feats = []
    for path in tqdm(img_paths, desc='Extracting', leave=False):
        img = Image.open(path).convert('RGB')
        x   = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            f = dino(x)           # CLS token: (1, 384)
        feats.append(f.squeeze(0).cpu())
    return torch.stack(feats)    # (N, 384)

# 全クラスの特徴量を抽出
all_feats  = []   # (N_total, 384)
all_labels = []   # (N_total,)
img_paths_by_class = {}

print('\n特徴量を抽出中...')
for cls_idx, cls_name in enumerate(CLASSES):
    paths = sorted((IMG_ROOT / cls_name).glob('*.jpg'))
    if not paths:
        print(f'  ⚠ {cls_name}: 画像なし → スキップ')
        continue
    img_paths_by_class[cls_name] = paths
    feats  = extract_features(paths)
    labels = torch.full((len(paths),), cls_idx, dtype=torch.long)
    all_feats.append(feats)
    all_labels.append(labels)
    print(f'  ✓ {cls_name:12s}: {len(paths)}枚 → {feats.shape}')

X = torch.cat(all_feats,  dim=0)   # (N, 384)
Y = torch.cat(all_labels, dim=0)   # (N,)
print(f'\n全特徴量: X={X.shape}  Y={Y.shape}')

# 特徴量の可視化 (t-SNE)
try:
    from sklearn.manifold import TSNE
    print('t-SNE で可視化中...')
    tsne  = TSNE(n_components=2, random_state=42, perplexity=min(5, len(X)-1))
    X_2d  = tsne.fit_transform(X.numpy())
    colors = plt.cm.tab10(np.linspace(0,1,N_CLASSES))
    fig, ax = plt.subplots(figsize=(8,6))
    for i, cls in enumerate(CLASSES):
        mask = Y.numpy() == i
        if mask.sum() == 0: continue
        ax.scatter(X_2d[mask,0], X_2d[mask,1],
                   c=[colors[i]], label=cls, s=60, alpha=0.8)
    ax.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=9)
    ax.set_title('DINOv2 特徴量 t-SNE (クラスが分離していれば学習成功の見込み大)',
                 color='#c8d8e8')
    plt.tight_layout(); plt.show()
except ImportError:
    print('sklearn未インストール: !pip install scikit-learn でt-SNEを可視化できます')

In [ ]:
# ============================================================
# セル 5: 分類ヘッドの学習
# ============================================================

class BuildingClassifier(nn.Module):
    """
    DINOv2 特徴ベクトル (384次元) → 建物クラス確率 (N_CLASSES次元)
    このモデルだけを学習する。DINOv2本体はfrozen。
    """
    def __init__(self, in_dim=DINO_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)  # logits (N, N_CLASSES)


# ── 学習データの分割 (80% train / 20% val) ──
torch.manual_seed(42)
n_total = len(X)
perm    = torch.randperm(n_total)
n_train = int(n_total * 0.8)
train_idx = perm[:n_train]
val_idx   = perm[n_train:]

X_train, Y_train = X[train_idx].to(DEVICE), Y[train_idx].to(DEVICE)
X_val,   Y_val   = X[val_idx].to(DEVICE),   Y[val_idx].to(DEVICE)
print(f'Train: {len(X_train)}件  Val: {len(X_val)}件')

# ── 学習 ──
clf       = BuildingClassifier().to(DEVICE)
optimizer = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

N_EPOCHS  = 200
best_val_acc = 0.0
train_losses, val_accs = [], []

for epoch in range(N_EPOCHS):
    # Train
    clf.train()
    logits = clf(X_train)
    loss   = criterion(logits, Y_train)
    optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
    train_losses.append(loss.item())

    # Val
    clf.eval()
    with torch.no_grad():
        val_logits = clf(X_val)
        val_pred   = val_logits.argmax(dim=1)
        val_acc    = (val_pred == Y_val).float().mean().item()
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.clone() for k, v in clf.state_dict().items()}

    if (epoch+1) % 50 == 0:
        print(f'Epoch {epoch+1:3d} | Loss:{loss.item():.4f} | '
              f'ValAcc:{val_acc:.1%} | Best:{best_val_acc:.1%}')

# 最良モデルを復元
clf.load_state_dict(best_state)
print(f'\n✓ 学習完了  最高 Val Acc: {best_val_acc:.1%}')

# 学習曲線
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(10,3))
fig.patch.set_facecolor('#0a0c10')
ax1.plot(train_losses, color='#ff5050', lw=1.5); ax1.set_title('Train Loss', color='#c8d8e8')
ax2.plot(val_accs,     color='#00ddb4', lw=1.5); ax2.set_title('Val Accuracy', color='#c8d8e8')
ax2.set_ylim(0,1)
for ax in [ax1,ax2]: ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# セル 6: 混同行列で分類性能を確認
# ============================================================
clf.eval()
with torch.no_grad():
    all_pred = clf(X.to(DEVICE)).argmax(dim=1).cpu()

from collections import Counter
conf = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for true, pred in zip(Y.numpy(), all_pred.numpy()):
    conf[true, pred] += 1

fig, ax = plt.subplots(figsize=(8,7))
fig.patch.set_facecolor('#0a0c10')
im = ax.imshow(conf, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES, fontsize=9)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, str(conf[i,j]),
                ha='center', va='center', fontsize=10,
                color='white' if conf[i,j]>conf.max()*0.5 else '#c8d8e8')
ax.set_xlabel('Predicted', color='#c8d8e8')
ax.set_ylabel('True',      color='#c8d8e8')
ax.set_title('混同行列 (対角線上が正解)', color='#00ddb4', fontsize=12)
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

# クラス別精度
print('クラス別精度:')
for i, cls in enumerate(CLASSES):
    n   = conf[i].sum()
    acc = conf[i,i] / max(n, 1)
    bar = '█' * int(acc*20) + '░' * (20-int(acc*20))
    print(f'  {cls:12s} {bar} {acc:.0%} ({conf[i,i]}/{n})')

In [ ]:
# ============================================================
# セル 7: ONNX エクスポート
# ============================================================

clf.eval().cpu()

# ── 分類ヘッドを ONNX export ──
# 入力: DINOv2 CLS token (1, 384)
# 出力: クラス logits (1, N_CLASSES)

onnx_path = f'{ONNX_DIR}/building_classifier.onnx'
dummy = torch.zeros(1, DINO_DIM)

with torch.no_grad():
    torch.onnx.export(
        clf, dummy, onnx_path,
        input_names=['dino_feat'],
        output_names=['logits'],
        dynamic_axes={'dino_feat': {0:'batch'}, 'logits': {0:'batch'}},
        opset_version=17,
    )

# 外部ファイルを作らない
onnx_model = onnx.load(onnx_path)
onnx.save(onnx_model, onnx_path, save_as_external_data=False)
size_mb = os.path.getsize(onnx_path) / 1e6
print(f'✓ building_classifier.onnx ({size_mb:.2f}MB)')

# 推論テスト
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_np = np.zeros((1, DINO_DIM), dtype=np.float32)
logits   = sess.run(None, {'dino_feat': dummy_np})[0]
print(f'✓ 推論テスト OK  logits shape={logits.shape}')

# メタデータ保存
meta = {
    'classes':   CLASSES,
    'n_classes': N_CLASSES,
    'dino_dim':  DINO_DIM,
    'dino_model': 'dinov2_vits14',
    'input_name':  'dino_feat',
    'output_name': 'logits',
    'best_val_acc': best_val_acc,
    'n_train_images': int(n_total),
}
meta_path = onnx_path.replace('.onnx', '_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✓ building_classifier_meta.json 保存')

print()
print('='*50)
print('✅ Step 1.5 完了!')
print(f'   保存先: {ONNX_DIR}')
print(f'   - building_classifier.onnx')
print(f'   - building_classifier_meta.json')
print()
print('次のステップ:')
print('  step2_persona_train.ipynb を実行すると')
print('  このONNXを自動的に読み込んで学習に使います')

In [ ]:
# ============================================================
# セル 8: 自分の画像で追加学習 (オプション)
#
# 精度が低いクラスがある場合、
# 自分でGoogle Drive に写真を追加してから再実行する
#
# 追加方法:
#   /content/drive/MyDrive/mesa_persona/building_images/
#   └── gyudon/
#       ├── img_000.jpg  (既存)
#       ├── my_photo_01.jpg  ← ここに追加
#       └── my_photo_02.jpg
#
# 追加後にセル4→5→6→7を再実行するだけ
# ============================================================

# 現在の画像数を確認
print('現在の画像数 (追加後に確認):')
for cls in CLASSES:
    paths = list((IMG_ROOT / cls).glob('*.jpg')) + \
            list((IMG_ROOT / cls).glob('*.jpeg')) + \
            list((IMG_ROOT / cls).glob('*.png'))
    print(f'  {cls:12s}: {len(paths)}枚')
print()
print('精度の目安:')
print('  各クラス 5枚  → 70〜80% 程度')
print('  各クラス 20枚 → 85〜95% 程度')
print('  各クラス 50枚 → 95%+ 程度')